## LCEL 인터페이스


사용자 정의 체인을 가능한 쉽게 만들 수 있도록, [`Runnable`](https://api.python.langchain.com/en/stable/runnables/langchain_core.runnables.base.Runnable.html#langchain_core.runnables.base.Runnable) 프로토콜을 구현했습니다. 

`Runnable` 프로토콜은 대부분의 컴포넌트에 구현되어 있습니다.

이는 표준 인터페이스로, 사용자 정의 체인을 정의하고 표준 방식으로 호출하는 것을 쉽게 만듭니다.
표준 인터페이스에는 다음이 포함됩니다.

- [`stream`](#stream): 응답의 청크를 스트리밍합니다.
- [`invoke`](#invoke): 입력에 대해 체인을 호출합니다.
- [`batch`](#batch): 입력 목록에 대해 체인을 호출합니다.

비동기 메소드도 있습니다.

- [`astream`](#async-stream): 비동기적으로 응답의 청크를 스트리밍합니다.
- [`ainvoke`](#async-invoke): 비동기적으로 입력에 대해 체인을 호출합니다.
- [`abatch`](#async-batch): 비동기적으로 입력 목록에 대해 체인을 호출합니다.
- [`astream_log`](#async-stream-intermediate-steps): 최종 응답뿐만 아니라 발생하는 중간 단계를 스트리밍합니다.

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

False

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


LCEL 문법을 사용하여 chain 을 생성합니다.

In [3]:
# from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOllama 모델을 인스턴스화합니다.
model = ChatOllama(model="gemma4:26b", temperature=0.1)
# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")
# 프롬프트와 모델을 연결하여 대화 체인을 생성합니다.
chain = prompt | model | StrOutputParser()

## stream: 실시간 출력


이 함수는 `chain.stream` 메서드를 사용하여 주어진 토픽에 대한 데이터 스트림을 생성하고, 이 스트림을 반복하여 각 데이터의 내용(`content`)을 즉시 출력합니다. `end=""` 인자는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True` 인자는 출력 버퍼를 즉시 비우도록 합니다. 

In [4]:
# chain.stream 메서드를 사용하여 '멀티모달' 토픽에 대한 스트림을 생성하고 반복합니다.
for token in chain.stream({"topic": "멀티모달"}):
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

멀티모달은 텍스트, 이미지, 오디오, 비디오 등 서로 다른 형태의 데이터를 동시에 처리하고 이해하는 인공지능 기술을 의미합니다. 이는 인간이 시각, 청각 등 여러 감각을 사용하여 정보를 받아들이는 방식과 유사하게 작동합니다. 이를 통해 AI는 단순한 텍스트 응답을 넘어 이미지 생성이나 영상 분석과 같이 더욱 복합적이고 자연스러운 상호작용을 수행할 수 있습니다.

## invoke: 호출


`chain` 객체의 `invoke` 메서드는 주제를 인자로 받아 해당 주제에 대한 처리를 수행합니다.

In [5]:
# chain 객체의 invoke 메서드를 호출하고, 'ChatGPT'라는 주제로 딕셔너리를 전달합니다.
chain.invoke({"topic": "ChatGPT"})

'ChatGPT에 대한 설명입니다.\n\n1. ChatGPT는 OpenAI가 개발한 대화형 인공지능 서비스로, 대규모 언어 모델을 기반으로 작동합니다.\n2. 사용자의 질문에 자연스럽게 답변할 뿐만 아니라 글쓰기, 코딩, 번역 등 다양한 창의적 업무를 수행할 수 있습니다.\n3. 방대한 데이터를 학습하여 문맥을 이해하고 사람과 유사한 방식으로 대화를 이어가는 것이 특징입니다.'

## batch: 배치(단위 실행)


함수 `chain.batch`는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 `topic` 키의 값을 사용하여 일괄 처리를 수행합니다.

In [6]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])

['ChatGPT에 대한 설명입니다.\n\n1. ChatGPT는 OpenAI가 개발한 대화형 인공지능 서비스로, 대규모 언어 모델을 기반으로 작동합니다.\n2. 사용자의 질문에 자연스러운 문장으로 답변할 뿐만 아니라 글쓰기, 코딩, 번역, 요약 등 다양한 작업을 수행할 수 있습니다.\n3. 방대한 데이터를 학습하여 문맥을 이해하고 인간과 유사한 방식으로 소통하는 것이 가장 큰 특징입니다.',
 '인스타그램은 사진과 동영상을 중심으로 소통하는 대표적인 소셜 미디어 플랫폼입니다. 사용자는 게시물, 스토리, 릴스 등 다양한 기능을 통해 자신의 일상을 공유하고 전 세계 사람들과 연결될 수 있습니다. 현재는 개인적인 기록뿐만 아니라 브랜드 마케팅과 최신 트렌드를 파악하는 중요한 도구로도 널리 활용되고 있습니다.']

`max_concurrency` 매개변수를 사용하여 동시 요청 수를 설정할 수 있습니다

`config` 딕셔너리는 `max_concurrency` 키를 통해 동시에 처리할 수 있는 최대 작업 수를 설정합니다. 여기서는 최대 3개의 작업을 동시에 처리하도록 설정되어 있습니다.

In [7]:
chain.batch(
    [
        {"topic": "ChatGPT"},
        {"topic": "Instagram"},
        {"topic": "멀티모달"},
        {"topic": "프로그래밍"},
        {"topic": "머신러닝"},
    ],
    config={"max_concurrency": 3},
)

['ChatGPT에 대한 설명입니다.\n\n1. ChatGPT는 OpenAI가 개발한 대화형 인공지능 서비스로, 대규모 언어 모델을 기반으로 작동합니다.\n2. 사용자의 질문에 자연스러운 문장으로 답변할 뿐만 아니라 글쓰기, 코딩, 번역, 요약 등 다양한 작업을 수행할 수 있습니다.\n3. 방대한 데이터를 학습하여 문맥을 이해하고 인간과 유사한 방식으로 소통할 수 있다는 점이 가장 큰 특징입니다.',
 '인스타그램은 사진과 동영상을 중심으로 소통하는 대표적인 소셜 미디어 플랫폼입니다. 사용자는 다양한 필터와 편집 기능을 활용해 자신의 일상을 게시물, 스토리, 릴스 등의 형태로 공유할 수 있습니다. 시각적인 콘텐츠를 통해 전 세계 사람들과 연결되며, 최신 트렌드를 확인하고 발견하는 데 매우 유용합니다.',
 '멀티모달은 텍스트, 이미지, 오디오, 비디오 등 서로 다른 형태의 데이터를 동시에 처리하고 이해하는 인공지능 기술을 의미합니다. 이는 인간이 시각, 청각 등 여러 감각을 사용하여 정보를 받아들이는 방식과 유사하게 작동합니다. 이를 통해 AI는 단일 데이터 분석을 넘어 복합적인 맥락을 파악함으로써 더욱 정교하고 자연스러운 상호작용을 수행할 수 있습니다.',
 '프로그래밍은 컴퓨터가 특정 작업을 수행할 수 있도록 명령을 내리는 과정입니다. 이를 위해 파이썬이나 자바와 같은 프로그래밍 언어를 사용하여 컴퓨터가 이해할 수 있는 코드를 작성합니다. 결과적으로 프로그래밍을 통해 소프트웨어, 앱, 웹사이트 등 다양한 디지털 도구를 만들어 복잡한 문제를 해결할 수 있습니다.',
 '머신러닝은 컴퓨터가 명시적인 프로그래밍 없이 데이터를 통해 스스로 학습하는 인공지능의 한 분야입니다. 방대한 데이터를 분석하여 그 안에 숨겨진 패턴과 규칙을 찾아내는 것이 핵심 원리입니다. 이렇게 학습된 모델을 활용하면 새로운 데이터가 들어왔을 때 미래를 예측하거나 최선의 결정을 내릴 수 있습니다.']

## async stream: 비동기 스트림


함수 `chain.astream`은 비동기 스트림을 생성하며, 주어진 토픽에 대한 메시지를 비동기적으로 처리합니다.

비동기 for 루프(`async for`)를 사용하여 스트림에서 메시지를 순차적으로 받아오고, `print` 함수를 통해 메시지의 내용(`s.content`)을 즉시 출력합니다. `end=""`는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True`는 출력 버퍼를 강제로 비워 즉시 출력되도록 합니다.


In [8]:
# 비동기 스트림을 사용하여 'YouTube' 토픽의 메시지를 처리합니다.
async for token in chain.astream({"topic": "YouTube"}):
    # 메시지 내용을 출력합니다. 줄바꿈 없이 바로 출력하고 버퍼를 비웁니다.
    print(token, end="", flush=True)

유튜브는 구글이 운영하는 세계 최대의 동영상 공유 플랫폼입니다. 사용자는 다양한 주제의 영상을 시청하거나 직접 영상을 업로드하여 전 세계 사람들과 소통할 수 있습니다. 정보 검색부터 엔터테인먼트까지 폭넓은 콘텐츠를 제공하며, 현대 사회의 크리에이터 경제를 이끄는 핵심적인 역할을 합니다.

## async invoke: 비동기 호출


`chain` 객체의 `ainvoke` 메서드는 비동기적으로 주어진 인자를 사용하여 작업을 수행합니다. 여기서는 `topic`이라는 키와 `NVDA`(엔비디아의 티커) 라는 값을 가진 딕셔너리를 인자로 전달하고 있습니다. 이 메서드는 특정 토픽에 대한 처리를 비동기적으로 요청하는 데 사용될 수 있습니다.


In [9]:
# 비동기 체인 객체의 'ainvoke' 메서드를 호출하여 'NVDA' 토픽을 처리합니다.
my_process = chain.ainvoke({"topic": "NVDA"})

In [10]:
# 비동기로 처리되는 프로세스가 완료될 때까지 기다립니다.
await my_process

'엔비디아(NVDA)에 대한 설명입니다.\n\n1. 엔비디아는 그래픽 처리 장치(GPU) 설계 분야를 선도하는 세계적인 반도체 기업입니다.\n2. 초기에는 게이밍용 그래픽 카드로 유명했으나, 현재는 데이터 센터와 고성능 컴퓨팅 분야의 핵심 기술을 제공하고 있습니다.\n3. 특히 인공지능(AI) 연산에 필수적인 가속기를 공급하며 전 세계 AI 혁명을 이끄는 가장 중요한 기업 중 하나로 자리매김했습니다.'

## async batch: 비동기 배치


함수 `abatch`는 비동기적으로 일련의 작업을 일괄 처리합니다.

이 예시에서는 `chain` 객체의 `abatch` 메서드를 사용하여 `topic` 에 대한 작업을 비동기적으로 처리하고 있습니다.

`await` 키워드는 해당 비동기 작업이 완료될 때까지 기다리는 데 사용됩니다.


In [11]:
# 주어진 토픽에 대해 비동기적으로 일괄 처리를 수행합니다.
my_abatch_process = chain.abatch(
    [{"topic": "YouTube"}, {"topic": "Instagram"}, {"topic": "Facebook"}]
)

In [12]:
# 비동기로 처리되는 일괄 처리 프로세스가 완료될 때까지 기다립니다.
await my_abatch_process

['YouTube에 대한 설명입니다.\n\n1. YouTube는 사용자들이 동영상을 시청하고 직접 업로드할 수 있는 세계 최대의 동영상 공유 플랫폼입니다.\n2. 엔터테인먼트, 교육, 뉴스 등 매우 다양한 분야의 콘텐츠를 제공하여 전 세계 사람들에게 즐거움과 정보를 전달합니다.\n3. 또한, 구독과 라이브 스트리밍 기능을 통해 창작자와 시청자가 활발하게 소통하는 강력한 커뮤니티 역할을 합니다.',
 "인스타그램에 대한 설명입니다.\n\n1. 인스타그램은 사진과 동영상을 중심으로 일상을 공유하는 글로벌 소셜 미디어 플랫폼입니다.\n2. 다양한 필터와 편집 도구를 활용해 시각적으로 매력적인 콘텐츠를 누구나 쉽게 제작하고 게시할 수 있습니다.\n3. 최근에는 '릴스'와 같은 짧은 영상 기능을 통해 전 세계 사용자들과 더욱 역동적이고 몰입감 있게 소통하고 있습니다.",
 "페이스북은 전 세계 사람들이 친구, 가족과 소통할 수 있도록 연결해 주는 글로벌 소셜 네트워크 서비스(SNS)입니다. 사용자는 텍스트, 사진, 동영상 등 다양한 콘텐츠를 공유하고 댓글이나 '좋아요'를 통해 서로 교류할 수 있습니다. 현재는 메타(Meta)가 운영하고 있으며, 세계에서 가장 영향력 있는 디지털 소통 플랫폼 중 하나로 자리 잡고 있습니다."]

## Parallel: 병렬성

LangChain Expression Language가 병렬 요청을 지원하는 방법을 살펴봅시다.
예를 들어, `RunnableParallel`을 사용할 때, 각 요소를 병렬로 실행합니다.


`langchain_core.runnables` 모듈의 `RunnableParallel` 클래스를 사용하여 두 가지 작업을 병렬로 실행하는 예시를 보여줍니다.

`ChatPromptTemplate.from_template` 메서드를 사용하여 주어진 `country`에 대한 **수도** 와 **면적** 을 구하는 두 개의 체인(`chain1`, `chain2`)을 만듭니다.

이 체인들은 각각 `model`과 파이프(`|`) 연산자를 통해 연결됩니다. 마지막으로, `RunnableParallel` 클래스를 사용하여 이 두 체인을 `capital`와 `area`이라는 키로 결합하여 동시에 실행할 수 있는 `combined` 객체를 생성합니다.


In [13]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

`chain1.invoke()` 함수는 `chain1` 객체의 `invoke` 메서드를 호출합니다.

이때, `country`이라는 키에 `대한민국`라는 값을 가진 딕셔너리를 인자로 전달합니다.


In [14]:
# chain1 를 실행합니다.
chain1.invoke({"country": "대한민국"})

'대한민국의 수도는 **서울**입니다.'

이번에는 `chain2.invoke()` 를 호출합니다. `country` 키에 다른 국가인 `미국` 을 전달합니다.


In [15]:
# chain2 를 실행합니다.
chain2.invoke({"country": "미국"})

'미국의 총 면적은 약 **9,833,517 km²**입니다.\n\n이를 좀 더 자세히 나누면 다음과 같습니다:\n\n* **육지 면적:** 약 9,147,593 km²\n* **수역(물) 면적:** 약 685,924 km²\n\n미국은 세계에서 면적이 가장 넓은 나라 중 3위 또는 4위(러시아, 캐나다, 중국과의 순위는 기준에 따라 조금씩 다를 수 있음)에 해당할 정도로 매우 넓은 영토를 가지고 있습니다.'

`combined` 객체의 `invoke` 메서드는 주어진 `country`에 대한 처리를 수행합니다.

이 예제에서는 `대한민국`라는 주제를 `invoke` 메서드에 전달하여 실행합니다.


In [16]:
# 병렬 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 **서울**입니다.',
 'area': '대한민국의 면적은 약 **100,412 km²**입니다.\n\n(통계 자료나 기준에 따라 약간의 차이가 있을 수 있습니다.)'}

### 배치에서의 병렬 처리

병렬 처리는 다른 실행 가능한 코드와 결합될 수 있습니다.
배치와 병렬 처리를 사용해 보도록 합시다.


`chain1.batch` 함수는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 "topic" 키에 해당하는 값을 처리합니다. 이 예시에서는 "대한민국"와 "미국"라는 두 개의 토픽을 배치 처리하고 있습니다.


In [17]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 수도는 **서울**입니다.', '미국의 수도는 **워싱턴, D.C.**입니다.']

`chain2.batch` 함수는 여러 개의 딕셔너리를 리스트 형태로 받아, 일괄 처리(batch)를 수행합니다.

이 예시에서는 `대한민국`와 `미국`라는 두 가지 국가에 대한 처리를 요청합니다.


In [18]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 면적은 약 **100,412 km²**입니다.\n\n통계 자료나 측정 기준(해안선 변화, 간척 사업 등)에 따라 약간의 차이가 있을 수 있지만, 일반적으로 이 수치를 사용합니다. 참고로 이는 한반도 전체 면적(약 22만 km²)의 약 절반 정도에 해당합니다.',
 '미국의 총 면적은 약 **9,833,517 km²**입니다.\n\n이 면적에 대해 몇 가지 추가 정보를 드리자면 다음과 같습니다.\n\n1.  **세계 순위:** 면적 기준으로 세계에서 **3위 또는 4위** 정도의 넓은 나라입니다. (측정 기준에 따라 러시아, 캐나다, 중국과 순위가 바뀔 수 있습니다.)\n2.  **대한민국과의 비교:** 대한민국의 면적(약 10만 km²)과 비교하면 **약 98배** 정도 더 넓은 규모입니다.\n3.  **구성:** 이 면적에는 육지 면적뿐만 아니라 미국의 영해와 호수 등 수역 면적도 포함되어 있습니다.']

`combined.batch` 함수는 주어진 데이터를 배치로 처리하는 데 사용됩니다. 이 예시에서는 두 개의 딕셔너리 객체를 포함하는 리스트를 인자로 받아 각각 `대한민국`와 `미국` 두 나라에 대한 데이터를 배치 처리합니다.


In [19]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])

[{'capital': '대한민국의 수도는 **서울**입니다.',
  'area': '대한민국의 면적은 약 **100,412 km²**입니다.'},
 {'capital': '미국의 수도는 **워싱턴, D.C.**입니다.',
  'area': '미국의 총 면적은 약 **9,833,517 km²**입니다.\n\n이 면적에 대해 몇 가지 추가 정보를 드리자면 다음과 같습니다.\n\n1.  **세계 순위:** 면적 기준으로 세계에서 **3위 또는 4위** 정도의 넓은 나라입니다. (측정 기준에 따라 러시아, 캐나다, 중국과 순위가 다를 수 있습니다.)\n2.  **대한민국과의 비교:** 대한민국의 면적(약 10만 km²)과 비교하면 **약 100배** 정도 되는 매우 넓은 크기입니다.\n3.  **구성:** 이 면적에는 육지 면적뿐만 아니라 영해를 포함한 수역(물) 면적도 포함되어 있습니다.'}]